In [2]:
!pip install captum torchvision seaborn matplotlib pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 136.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incomp

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [2]:
import os, gc, glob, torch, numpy as np, pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from captum.attr import IntegratedGradients, GradientShap, LayerGradCam, Lime
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Device Config
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#Directory Definition

IMAGE_ROOT = "/content/drive/MyDrive/xai-stability-data/coco/train2017"
RESULT_DIR = "/content/drive/MyDrive/xai-stability-benchmark/results_coco"

os.makedirs(RESULT_DIR, exist_ok=True)

print(DEVICE)


cuda


In [4]:
# Lets do one perturbation per image - I'm using ROTATION - you can use any perturbation of your choice
PERTURBATION_TYPE = "rotation"
# Some Options you can use instead of rotation: "brightness", "noise", "translation", "jpeg"


In [5]:
#FASS Depends on 3 measures: 1) Spearman Score 2) SSIM 3) Top-K-Jaccard Similarity


#SSIM
def get_ssim_spatial(attr1, attr2):
    def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)
    a1, a2 = norm(attr1), norm(attr2)
    pool = nn.AvgPool2d(11, stride=1, padding=5)
    mu1, mu2 = pool(a1), pool(a2)
    sigma12 = pool(a1 * a2) - mu1 * mu2
    sigma1_sq = pool(a1 * a1) - mu1 * mu1
    sigma2_sq = pool(a2 * a2) - mu2 * mu2
    c1, c2 = 0.01**2, 0.03**2
    ssim_map = ((2 * mu1 * mu2 + c1) * (2 * sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_map.view(attr1.shape[0], -1).mean(dim=1)

# Spearman
def get_spearman_rank(attr1, attr2):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    r1 = torch.argsort(torch.argsort(v1, dim=1), dim=1).float()
    r2 = torch.argsort(torch.argsort(v2, dim=1), dim=1).float()
    mu1, mu2 = r1.mean(dim=1, keepdim=True), r2.mean(dim=1, keepdim=True)
    num = ((r1 - mu1) * (r2 - mu2)).sum(dim=1)
    den = torch.sqrt(((r1 - mu1)**2).sum(dim=1) * ((r2 - mu2)**2).sum(dim=1)) + 1e-8
    return (num / den + 1) / 2

# Top-K-Jaccard
def get_top_k_jaccard(attr1, attr2, k=100):
    b = attr1.shape[0]
    v1, v2 = attr1.view(b, -1), attr2.view(b, -1)
    _, idx1 = torch.topk(v1, k, dim=1)
    _, idx2 = torch.topk(v2, k, dim=1)
    jaccards = []
    for i in range(b):
        mask = torch.isin(idx1[i], idx2[i])
        intersection = mask.sum().float()
        union = 2 * k - intersection
        jaccards.append(intersection / union)
    return torch.tensor(jaccards).to(DEVICE)

In [6]:
def calculate_fass(attr1, attr2):
    return (get_ssim_spatial(attr1, attr2) + get_spearman_rank(attr1, attr2) + get_top_k_jaccard(attr1, attr2)) / 3

In [7]:
# COCO Dataset
class FASSCOCODataset(Dataset):
    def __init__(self, root, image_list, p_type="rotation"):
        self.root, self.image_list, self.p_type = root, image_list, p_type
        self.base_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        self.norm_tf = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    def __len__(self): return len(self.image_list)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.image_list[idx])).convert("RGB")
        img = TF.resize(img, (224, 224))
        orig = self.base_tf(img)
        if self.p_type == "rotation":
            p = self.norm_tf(transforms.ToTensor()(TF.rotate(img, 5)))
        elif self.p_type == "brightness":
            p = self.norm_tf(transforms.ToTensor()(TF.adjust_brightness(img, 1.1)))
        elif self.p_type == "noise":
            raw = transforms.ToTensor()(img)
            noisy = torch.clamp(raw + (torch.randn_like(raw) * 0.01), 0, 1)
            p = self.norm_tf(noisy)
        elif self.p_type == "translation":
            p = self.norm_tf(transforms.ToTensor()(TF.affine(img, angle=0, translate=(10, 0), scale=1.0, shear=0)))
        elif self.p_type == "jpeg":
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            buf.seek(0)
            p = self.norm_tf(transforms.ToTensor()(Image.open(buf).convert("RGB")))
        return torch.stack([orig, p]), idx


In [8]:
# Defining Models:
# 4 models are used => 1) ResNet50, 2) DenseNet121, 3) ConvNextTiny, and 4) VIT-b-16

def prepare_models():
    models_dict = {"resnet50": models.resnet50(weights="IMAGENET1K_V1"), "densenet121": models.densenet121(weights="IMAGENET1K_V1"),
                   "convnext_tiny": models.convnext_tiny(weights="IMAGENET1K_V1"), "vit_b_16": models.vit_b_16(weights="IMAGENET1K_V1")}
    for name, m in models_dict.items():
        if "resnet" in name: m.fc = nn.Linear(m.fc.in_features, 80)
        elif "dense" in name: m.classifier = nn.Linear(m.classifier.in_features, 80)
        elif "convnext" in name: m.classifier[2] = nn.Linear(m.classifier[2].in_features, 80)
        elif "vit" in name: m.heads.head = nn.Linear(m.heads.head.in_features, 80)
        for module in m.modules():
            if isinstance(module, nn.ReLU): module.inplace = False
        m.to(DEVICE).eval()
    return models_dict

In [9]:
def run_benchmark(name, model, images):
    path = os.path.join(RESULT_DIR, f"{name}_fass_{PERTURBATION_TYPE}.csv")
    start = 0
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            if not df.empty:
                start = df["image_index"].max() + 1
        except: pass
    if start >= len(images):
        print(f"SKIP: {name} + {PERTURBATION_TYPE} already complete ({start} images).")
        return
    print(f"RESUMING: {name} + {PERTURBATION_TYPE} from image {start}/{len(images)}")
    loader = DataLoader(FASSCOCODataset(IMAGE_ROOT, images[start:], PERTURBATION_TYPE), batch_size=128)
    ig, gs, lime = IntegratedGradients(model), GradientShap(model), Lime(model)
    if name == "resnet50": target_layer = model.layer4[-1]
    elif name == "densenet121": target_layer = model.features[-1]
    elif name == "convnext_tiny": target_layer = model.features[-1]
    elif name == "vit_b_16": target_layer = model.encoder.layers[-1]
    gcam = LayerGradCam(model, target_layer)
    results = []
    for stack, idxs in tqdm(loader, desc=name):
        orig, p = stack[:,0].to(DEVICE).requires_grad_(True), stack[:,1].to(DEVICE)
        with torch.no_grad():
            t = torch.argmax(model(orig), 1)
            mask = (t == torch.argmax(model(p), 1))
        if not mask.any(): continue
        res_ig = calculate_fass(ig.attribute(orig[mask], t[mask]), ig.attribute(p[mask], t[mask]))
        res_gs = calculate_fass(gs.attribute(orig[mask], t[mask], n_samples=5, baselines=torch.zeros_like(orig[mask])), gs.attribute(p[mask], t[mask], n_samples=5, baselines=torch.zeros_like(p[mask])))
        res_gc = calculate_fass(TF.resize(gcam.attribute(orig[mask], t[mask]), (224,224)), TF.resize(gcam.attribute(p[mask], t[mask]), (224,224)))
        res_lm = calculate_fass(lime.attribute(orig[mask], t[mask], n_samples=50), lime.attribute(p[mask], t[mask], n_samples=50))
        for i, val in enumerate(idxs[mask]):
            results.append({"image_index": val.item(), "ig_fass": res_ig[i].item(), "shap_fass": res_gs[i].item(), "gradcam_fass": res_gc[i].item(), "lime_fass": res_lm[i].item()})
        if len(results) >= 128:
            pd.DataFrame(results).to_csv(path, mode='a', header=not os.path.exists(path), index=False)
            results = []; torch.cuda.empty_cache()
    if results: pd.DataFrame(results).to_csv(path, mode='a', header=not os.path.exists(path), index=False)


In [13]:
m_dict = prepare_models()
imgs = sorted([f for f in os.listdir(IMAGE_ROOT) if f.endswith(".jpg")])[:40000]
model_names = list(m_dict.keys())
total_models = len(model_names)

print(f"\n{'='*60}")
print(f"FASS BENCHMARK — Perturbation: {PERTURBATION_TYPE}")
print(f"Images: {len(imgs)} | Models: {total_models} | XAI Methods: 4")
print(f"{'='*60}\n")

for model_idx, (n, m) in enumerate(m_dict.items(), 1):
    print(f"\n>>> [{model_idx}/{total_models}] Checking {n} + {PERTURBATION_TYPE}...")
    check_path = os.path.join(RESULT_DIR, f"{n}_fass_{PERTURBATION_TYPE}.csv")
    if os.path.exists(check_path):
        try:
            df_check = pd.read_csv(check_path)
            if not df_check.empty and df_check["image_index"].max() + 1 >= len(imgs):
                print(f"    SKIP: Already complete ({len(df_check)} images saved).")
                del m; gc.collect(); torch.cuda.empty_cache()
                continue
            else:
                print(f"    PARTIAL: {len(df_check)} images found. Resuming...")
        except: pass
    else:
        print(f"    NEW: No prior results. Starting from scratch.")
    run_benchmark(n, m, imgs)
    print(f"    DONE: {n} + {PERTURBATION_TYPE} complete.")
    del m; gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"ALL MODELS COMPLETE for perturbation: {PERTURBATION_TYPE}")
print(f"Results saved to: {RESULT_DIR}")
print(f"{'='*60}")


FASS BENCHMARK — Perturbation: rotation
Images: 27217 | Models: 4 | XAI Methods: 4


>>> [1/4] Checking resnet50 + rotation...
    NEW: No prior results. Starting from scratch.
RESUMING: resnet50 + rotation from image 0/27217


resnet50:   0%|          | 0/213 [02:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
XAI_METHODS = {"ig_fass": "IG", "shap_fass": "SHAP", "gradcam_fass": "GradCAM", "lime_fass": "LIME"}
os.makedirs(os.path.join(RESULT_DIR, "figures"), exist_ok=True)

In [ ]:
def load_all_results(folder):
    all_files = glob.glob(os.path.join(folder, "*_fass_*.csv"))
    if not all_files:
        print("No result files found.")
        return None
    summary_list = []
    for file in all_files:
        filename = os.path.basename(file)
        parts = filename.replace(".csv", "").split("_fass_")
        model_name = parts[0]
        perturbation = parts[1]
        df = pd.read_csv(file)
        metrics = df.filter(like='_fass').mean().to_dict()
        stds = df.filter(like='_fass').std().to_dict()
        summary_list.append({
            'model': model_name, 'perturbation': perturbation, 'n_images': len(df),
            **metrics, **{k.replace('_fass', '_std'): v for k, v in stds.items()}
        })
    summary_df = pd.DataFrame(summary_list)
    summary_df.to_csv(os.path.join(folder, "FINAL_FASS_SUMMARY.csv"), index=False)
    print(f"Summary saved. {len(summary_df)} entries across {summary_df['model'].nunique()} models, {summary_df['perturbation'].nunique()} perturbations.")
    return summary_df

In [ ]:
def generate_latex_tables(summary_df):
    for model in summary_df['model'].unique():
        mdf = summary_df[summary_df['model'] == model].set_index('perturbation')
        rows = []
        for pert in mdf.index:
            row = {'Perturbation': pert}
            for col, label in XAI_METHODS.items():
                mean = mdf.loc[pert, col]
                std = mdf.loc[pert, col.replace('_fass', '_std')]
                row[label] = f"{mean:.3f} $\\pm$ {std:.3f}"
            rows.append(row)
        tex_df = pd.DataFrame(rows).set_index('Perturbation')
        avg_row = {'Perturbation': '\\textbf{Average}'}
        for col, label in XAI_METHODS.items():
            avg_row[label] = f"{mdf[col].mean():.3f}"
        tex_df = pd.concat([tex_df, pd.DataFrame([avg_row]).set_index('Perturbation')])
        print(f"\n% === {model} ===")
        print(tex_df.to_latex(escape=False, column_format='l' + 'c' * len(XAI_METHODS)))

In [ ]:
def generate_grand_summary_latex(summary_df):
    rows = []
    for model in summary_df['model'].unique():
        mdf = summary_df[summary_df['model'] == model]
        row = {'Model': model}
        for col, label in XAI_METHODS.items():
            row[label] = f"{mdf[col].mean():.3f}"
        rows.append(row)
    tex_df = pd.DataFrame(rows).set_index('Model')
    print("\n% === Grand Summary: Model x XAI Method ===")
    print(tex_df.to_latex(escape=False, column_format='l' + 'c' * len(XAI_METHODS)))


In [ ]:
def plot_per_model_heatmaps(summary_df):
    for model in summary_df['model'].unique():
        mdf = summary_df[summary_df['model'] == model].set_index('perturbation')
        heat_data = mdf[[c for c in XAI_METHODS.keys()]].rename(columns=XAI_METHODS)
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.3, vmax=1.0,
                    linewidths=0.5, ax=ax, cbar_kws={'label': 'FASS Score'})
        ax.set_title(f"FASS Stability — {model}", fontsize=13, fontweight='bold')
        ax.set_ylabel("Perturbation Type"); ax.set_xlabel("XAI Method")
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_DIR, "figures", f"heatmap_{model}.pdf"), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(RESULT_DIR, "figures", f"heatmap_{model}.png"), dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
def plot_grand_heatmap(summary_df):
    rows = []
    for model in summary_df['model'].unique():
        mdf = summary_df[summary_df['model'] == model]
        row = {label: mdf[col].mean() for col, label in XAI_METHODS.items()}
        row['Model'] = model
        rows.append(row)
    grand_df = pd.DataFrame(rows).set_index('Model')
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(grand_df, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.3, vmax=1.0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'FASS Score'})
    ax.set_title("FASS Stability — All Models (Avg. Across Perturbations)", fontsize=13, fontweight='bold')
    ax.set_ylabel("Model Architecture"); ax.set_xlabel("XAI Method")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "figures", "heatmap_grand.pdf"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(RESULT_DIR, "figures", "heatmap_grand.png"), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_grouped_bar(summary_df):
    for model in summary_df['model'].unique():
        mdf = summary_df[summary_df['model'] == model]
        perts = mdf['perturbation'].values
        x = np.arange(len(perts))
        width = 0.18
        fig, ax = plt.subplots(figsize=(10, 5))
        for i, (col, label) in enumerate(XAI_METHODS.items()):
            ax.bar(x + i * width, mdf[col].values, width, label=label)
        ax.set_xlabel("Perturbation Type"); ax.set_ylabel("FASS Score")
        ax.set_title(f"FASS by XAI Method & Perturbation — {model}", fontweight='bold')
        ax.set_xticks(x + width * 1.5); ax.set_xticklabels(perts, rotation=15)
        ax.legend(); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_DIR, "figures", f"barplot_{model}.pdf"), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(RESULT_DIR, "figures", f"barplot_{model}.png"), dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
summary = load_all_results(RESULT_DIR)
if summary is not None:
    generate_latex_tables(summary)
    generate_grand_summary_latex(summary)
    plot_per_model_heatmaps(summary)
    plot_grand_heatmap(summary)
    plot_grouped_bar(summary)
